# Hourly BTC Market Microstructure

This variant adds downside risk, trend indicators, and exchange microstructure features. This walk-forward backtest predicts next-hour BTC/USDT price direction. It uses completed hourly candles from the public Binance spot API, including quote volume, trade counts, and taker-buy volume. Simulated payoffs are a fixed gain or loss of the position size based on whether the prediction is correct. Starting from the configured dates can take substantial time. Set END_DATE in the run cell to restrict the historical window. [Binance spot market data documentation](https://developers.binance.com/en/docs/catalog/core-trading-spot-trading/api/rest-api/market)



In [ ]:
from _binance_data import download_hourly_candles

import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit
from arch import arch_model
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# SETTINGS
# ==========================================
SYMBOL = "BTCUSDT"

BASE_RISK = 0.01
MAX_POS = 0.11
VOL_GRID = [0.005, 0.010, 0.015, 0.020, 0.025, 0.030]
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ==========================================
# BINANCE DATA DOWNLOAD
# ==========================================

# ==========================================
# GJR-GARCH
# ==========================================
def add_gjr_garch_feature_1h(df_h):
    """Calculate a rolling GJR-GARCH(1,1) forecast using past observations."""
    print("--> Calculating rolling 6-hour GJR-GARCH...")
    
    df_6h = df_h['log_ret'].resample('6h').sum().dropna()
    print(f"   Available 6-hour periods: {len(df_6h)}")
    
    window = 28
    
    if len(df_6h) <= window:
        print(f"    Insufficient data 6h. Using historical volatility.")
        df_h['GJR_var_6h'] = (df_h['log_ret']**2).rolling(6).mean().shift(1)
        return df_h

    # Calcular forecasts
    gjr_forecasts = []
    dates = df_6h.index

    for i in range(window, len(df_6h)):
        train = df_6h.iloc[i-window:i]
        
        try:
            model = arch_model(train * 100, mean='Zero', vol='GARCH', 
                              p=1, o=1, q=1, dist='skewt')
            res = model.fit(disp="off", show_warning=False)
            fcast = res.forecast(horizon=1).variance.iloc[-1, 0]
        except:
            fcast = train.var() * 100
        
        valid_from = dates[i]
        valid_until = dates[i] + pd.Timedelta('6h')
        gjr_forecasts.append((valid_from, valid_until, fcast))


    df_h['GJR_var_6h'] = np.nan
    for valid_from, valid_until, fcast in gjr_forecasts:
        mask = (df_h.index >= valid_from) & (df_h.index < valid_until)
        df_h.loc[mask, 'GJR_var_6h'] = fcast
    

    hist_vol = (df_h['log_ret']**2).rolling(6).mean().shift(1)
    df_h['GJR_var_6h'] = df_h['GJR_var_6h'].fillna(hist_vol)
    
    nans = df_h['GJR_var_6h'].isna().sum()
    print(f"   NaNs finales GJR: {nans}")
    
    return df_h

# ==========================================
# HAR FEATURES
# ==========================================
def add_har_features_1h(df_h):
    """HAR con ventanas 1h, 6h shifteadas."""
    print("--> Calculating hourly HAR features...")
    
    r = df_h['log_ret']
    rv = r**2
    rs_pos = (r.clip(lower=0))**2
    rs_neg = (r.clip(upper=0))**2

    # Jump component (Bipower Variation)
    mu_1 = np.sqrt(2/np.pi)
    abs_r = r.abs()
    bpv_1h = (abs_r * abs_r.shift(1)).rolling(1).sum() * (mu_1 ** -2)
    jump_1h = np.maximum(rv.rolling(1).mean() - bpv_1h, 0)

    df_h['HAR_RV_1_1'] = rv.shift(1)
    df_h['HAR_RV_1_6'] = rv.rolling(6).mean().shift(1)
    
    df_h['HAR_RSpos_1_1'] = rs_pos.shift(1)
    df_h['HAR_RSpos_1_6'] = rs_pos.rolling(6).mean().shift(1)
    
    df_h['HAR_RSneg_1_1'] = rs_neg.shift(1)
    df_h['HAR_RSneg_1_6'] = rs_neg.rolling(6).mean().shift(1)
    
    df_h['HAR_Jump_1_1'] = jump_1h.shift(1)

    return df_h

# ==========================================
# RSI MEJORADO (WILDER)
# ==========================================
def calculate_rsi(series, periods=14):
    """RSI tradicional con suavizado Wilder."""
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).ewm(alpha=1/periods, adjust=False).mean()
    loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/periods, adjust=False).mean()
    rs = gain / (loss + 1e-9)
    rsi = 100 - 100/(1 + rs)
    return rsi.shift(1)

# ==========================================
# FEATURE ENGINEERING
# ==========================================
def prepare_features_1h(df):
    """Pipeline completo para 1H sin look-ahead bias."""
    print("--> Preparando features 1H...")
    print(f"[1] df inicial: {len(df)} rows")
    
    if len(df) < 2000:
        raise ValueError(f"Insufficient data: {len(df)} candles. A minimum of 2,000 candles is required.")
    
    df['log_ret'] = np.log(df['Close'] / df['Close'].shift(1))
    df = df.dropna()
    print(f"[2] After log_ret: {len(df)} rows")

    # GJR-GARCH y HAR
    df = add_gjr_garch_feature_1h(df)
    print(f"[3] After GJR: {len(df)}, NaN GJR: {df['GJR_var_6h'].isna().sum()}")
    
    df = add_har_features_1h(df)
    print(f"[4] After HAR: {len(df)}")

    # VOLATILITY
    df['RV_1h'] = (df['log_ret']**2).shift(1)
    df['RV_6h'] = df['log_ret'].rolling(6).std().shift(1) * np.sqrt(6)
    df['RV_24h'] = df['log_ret'].rolling(24).std().shift(1) * np.sqrt(24)
    
    # RSI MEJORADO
    df['RSI'] = calculate_rsi(df['Close'], periods=14)
    
    # VOLUME SHOCK
    df['Vol_Shock'] = (df['Volume'].shift(1) / 
                       (df['Volume'].rolling(12).mean().shift(1) + 1e-9))
    
    # KURTOSIS
    df['Kurt_6h'] = df['log_ret'].rolling(6).kurt().shift(1)
    
    # CANDLESTICK
    df['body_rel'] = ((df['Close'].shift(1) - df['Open'].shift(1)) / 
                       df['Open'].shift(1))
    df['wick_upper'] = ((df['High'].shift(1) - 
                         np.maximum(df['Open'].shift(1), df['Close'].shift(1))) / 
                        df['Open'].shift(1))
    df['wick_lower'] = ((np.minimum(df['Open'].shift(1), df['Close'].shift(1)) - 
                         df['Low'].shift(1)) / 
                        df['Open'].shift(1))
    
    # GARMAN-KLASS VOLATILITY (superior a close-to-close)
    log_hl = np.log(df['High'] / df['Low'])
    log_co = np.log(df['Close'] / df['Open'])
    df['GK_vol'] = (0.5 * log_hl**2 - (2*np.log(2)-1) * log_co**2).shift(1)
    

    neg_returns = df['log_ret'].clip(upper=0)
    df['Downside_Dev'] = neg_returns.rolling(12).std().shift(1)
    

    change = (df['Close'] - df['Close'].shift(10)).abs()
    volatility = df['Close'].diff().abs().rolling(10).sum()
    df['KER'] = (change / (volatility + 1e-9)).shift(1)
    
    # ADX (trend strength)
    high_diff = df['High'].diff()
    low_diff = -df['Low'].diff()
    plus_dm = np.where((high_diff > low_diff) & (high_diff > 0), high_diff, 0)
    minus_dm = np.where((low_diff > high_diff) & (low_diff > 0), low_diff, 0)
    
    tr1 = df['High'] - df['Low']
    tr2 = (df['High'] - df['Close'].shift()).abs()
    tr3 = (df['Low'] - df['Close'].shift()).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    
    atr = tr.rolling(14).mean()
    plus_di = 100 * pd.Series(plus_dm, index=df.index).rolling(14).mean() / atr
    minus_di = 100 * pd.Series(minus_dm, index=df.index).rolling(14).mean() / atr
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-9)
    df['ADX'] = dx.rolling(14).mean().shift(1)
    

    ma_20 = df['Close'].rolling(20).mean()
    df['Dist_MA'] = ((df['Close'] - ma_20) / ma_20).shift(1)
    
    # MICROESTRUCTURA
    df['taker_buy_ratio'] = (df['taker_base'] / (df['Volume'] + 1e-9)).shift(1)
    df['avg_trade_size'] = (df['quote_vol'] / (df['trades'] + 1e-9)).shift(1)
    df['num_trades'] = df['trades'].shift(1)
    df['hl_spread'] = ((df['High'] - df['Low']) / df['Close']).shift(1)
    
    # TARGET & RETURN
    next_close = df['Close'].shift(-1)
    df['target'] = (next_close > df['Close']).astype(int)
    df = df.loc[next_close.notna()].copy()
    df['ret'] = df['Close'].pct_change()
    
    # DROPNA
    feature_cols = [
        'RV_1h', 'RV_6h', 'RV_24h', 'RSI', 'Vol_Shock', 
        'Kurt_6h', 'body_rel', 'wick_upper', 'wick_lower', 'GJR_var_6h',
        'HAR_RV_1_1', 'HAR_RV_1_6', 'HAR_RSpos_1_1', 'HAR_RSpos_1_6',
        'HAR_RSneg_1_1', 'HAR_RSneg_1_6', 'HAR_Jump_1_1',
        'taker_buy_ratio', 'avg_trade_size', 'num_trades', 'hl_spread',
        'GK_vol', 'Downside_Dev', 'KER', 'ADX', 'Dist_MA'
    ]
    
    print(f"[5] Missing values before cleanup:")
    for col in feature_cols:
        nans = df[col].isna().sum()
        if nans > 0:
            print(f"   {col}: {nans} NaNs ({nans/len(df)*100:.1f}%)")
    
    total_antes = len(df)
    df = df.dropna(subset=feature_cols)
    print(f"[6] After dropna: {len(df)} rows (removed {total_antes - len(df)})")
    
    return df


FEATURES_1H = [
    'RV_1h', 'RV_6h', 'RV_24h',
    'RSI', 'Vol_Shock', 'Kurt_6h',
    'body_rel', 'wick_upper', 'wick_lower',
    'GJR_var_6h',
    'HAR_RV_1_1', 'HAR_RV_1_6',
    'HAR_RSpos_1_1', 'HAR_RSpos_1_6',
    'HAR_RSneg_1_1', 'HAR_RSneg_1_6',
    'HAR_Jump_1_1',
    'taker_buy_ratio', 'avg_trade_size', 'num_trades', 'hl_spread',
    'GK_vol', 'Downside_Dev', 'KER', 'ADX', 'Dist_MA'
]

# ==========================================
# BACKTEST
# ==========================================
class Binance_Backtester:
    def __init__(self, df, features):
        self.df = df
        self.features = features
        self.results = []
        self.BASE_RISK = BASE_RISK
        self.MAX_POS = MAX_POS
        self.VOL_GRID = VOL_GRID
        
        total_candles = len(df)
        if total_candles < 200:
            raise ValueError(f"Insufficient data: {total_candles} candles.")
        
        if total_candles >= 1500:
            self.TRAIN_SIZE = 1000
            self.TEST_SIZE = 180
        elif total_candles >= 800:
            self.TRAIN_SIZE = 500
            self.TEST_SIZE = 100
        else:
            self.TRAIN_SIZE = int(total_candles * 0.6)
            self.TEST_SIZE = int(total_candles * 0.15)
        
        print(f" Dataset: {total_candles} hourly candles")
        print(f"   Train Size: {self.TRAIN_SIZE}")
        print(f"   Test Size: {self.TEST_SIZE}")
        
    def optimize_target_vol(self, X_train, y_train, vol_series):
        """Optimiza target vol usando split train/validation."""

        split_point = int(len(X_train) * 0.75)
        
        X_model = X_train.iloc[:split_point]
        y_model = y_train.iloc[:split_point]
        
        X_val = X_train.iloc[split_point:]
        y_val = y_train.iloc[split_point:]
        vol_val = vol_series.iloc[split_point:]
        

        cv = TimeSeriesSplit(n_splits=3)
        model = xgb.XGBClassifier(
            n_estimators=100, max_depth=3, eval_metric='logloss',
            n_jobs=-1, random_state=RANDOM_SEED
        )
        calib = CalibratedClassifierCV(model, method='isotonic', cv=cv)
        calib.fit(X_model, y_model)
        

        probs = calib.predict_proba(X_val)[:, 1]
        
        best_sharpe = -999
        best_vol = 0.015
        
        for v_cand in self.VOL_GRID:
            pnls = []
            for p, v, real in zip(probs, vol_val, y_val):
                signal = 0
                if p > 0.51: signal = 1
                elif p < 0.49: signal = -1
                if signal == 0:
                    pnls.append(0)
                    continue
                conviction = max(0, (p - 0.50) * 2)
                vol_scalar = np.clip(v_cand / max(v, 0.001), 0.5, 2.0)
                size = min(self.BASE_RISK * conviction * 10 * vol_scalar, self.MAX_POS)
                res = size if ((signal==1 and real==1) or (signal==-1 and real==0)) else -size
                pnls.append(res)
            
            if len(pnls) == 0:
                continue
            sharpe = np.mean(pnls) / (np.std(pnls) + 1e-9)
            if sharpe > best_sharpe:
                best_sharpe = sharpe
                best_vol = v_cand
        
        return best_vol
    
    def run_walk_forward(self):
        print(f"--> Starting expanding-window walk-forward validation...")
        
        total_candles = len(self.df)
        total_steps = (total_candles - self.TRAIN_SIZE) // self.TEST_SIZE
        
        if total_steps < 1:
            print(f" Only {total_steps} full folds are available. Using one split.")
            total_steps = 1
        
        print(f"   {total_steps} walk-forward folds with expanding training windows")
        
        for i in range(total_steps):
            end_train = self.TRAIN_SIZE + i * self.TEST_SIZE
            start_train = 0
            
            start_test = end_train
            end_test = start_test + self.TEST_SIZE
            if end_test > total_candles:
                end_test = total_candles
            
            X_train = self.df[self.features].iloc[start_train:end_train]
            y_train = self.df['target'].iloc[start_train:end_train]
            vol_train = self.df['RV_1h'].iloc[start_train:end_train]
            
            X_test = self.df[self.features].iloc[start_test:end_test]
            y_test = self.df['target'].iloc[start_test:end_test]
            vol_test = self.df['RV_1h'].iloc[start_test:end_test]
            
            if len(X_test) == 0:
                break
            

            best_vol = self.optimize_target_vol(X_train, y_train, vol_train)
            

            cv = TimeSeriesSplit(n_splits=3)
            base = xgb.XGBClassifier(
                n_estimators=200, learning_rate=0.03, max_depth=4,
                subsample=0.7, colsample_bytree=0.7,
                n_jobs=-1, random_state=RANDOM_SEED
            )
            model = CalibratedClassifierCV(base, method='isotonic', cv=cv)
            model.fit(X_train, y_train)
            
            probs_test = model.predict_proba(X_test)[:, 1]
            
            for j in range(len(X_test)):
                p = probs_test[j]
                v = vol_test.iloc[j]
                real = y_test.iloc[j]
                idx = X_test.index[j]
                
                signal = 0
                if p > 0.53: signal = 1
                elif p < 0.47: signal = -1
                
                size = 0
                if signal != 0:
                    conviction = max(0, (p - 0.50) * 2)
                    vol_scalar = np.clip(best_vol / max(v, 0.001), 0.5, 2.0)
                    size = min(self.BASE_RISK * conviction * 10 * vol_scalar, self.MAX_POS)
                
                pnl = size if ((signal==1 and real==1) or (signal==-1 and real==0)) else -size
                if signal == 0:
                    pnl = 0
                
                self.results.append({
                    'Date': idx,
                    'Prob': p,
                    'Vol': v,
                    'Target_Vol': best_vol,
                    'Size': size,
                    'Signal': signal,
                    'Outcome': real,
                    'PnL': pnl
                })
            
            print(f"   Step {i+1}/{total_steps}: Train hasta {self.df.index[end_train-1]}, "
                f"Target Vol={best_vol*100:.1f}%, Test candles={len(X_test)}")
        
        if len(self.results) == 0:
            print(" No results were generated.")
            return pd.DataFrame()
        
        return pd.DataFrame(self.results).set_index('Date')

# ==========================================

# ==========================================


In [ ]:
START_DATE = "2017-08-01"
END_DATE = None  # Exclusive UTC endpoint. None uses the latest completed hour.

df_raw = download_hourly_candles(SYMBOL, START_DATE, END_DATE)
df = prepare_features_1h(df_raw)

try:
    backtester = Binance_Backtester(df, FEATURES_1H)
    results = backtester.run_walk_forward()

    if len(results) > 0:
        results['Equity'] = (1 + results['PnL']).cumprod()
        results['BuyHold'] = (1 + df.loc[results.index, 'ret']).cumprod()

        trades = results[results['Size'] > 0]
        win_rate = len(trades[trades['PnL'] > 0]) / len(trades) * 100 if len(trades) > 0 else 0

        print("\n" + "="*50)
        print("BINANCE BACKTEST RESULTS")
        print("="*50)
        print(f"Total Candles Evaluated:  {len(results)}")
        print(f"Simulated Trades: {len(trades)} ({len(trades)/len(results)*100:.1f}%)")
        print(f"Win Rate:               {win_rate:.2f}%")
        print(f"Strategy Return:     {(results['Equity'].iloc[-1]-1)*100:.2f}%")
        print(f"Buy-and-Hold Return:     {(results['BuyHold'].iloc[-1]-1)*100:.2f}%")
        print(f"Average Target Volatility:    {results['Target_Vol'].mean()*100:.2f}%")
        print(f"Average Position Size:          {trades['Size'].mean()*100:.2f}%" if len(trades) > 0 else "N/A")

        # Risk metrics
        equity = results['Equity']
        pnl = results['PnL']
        cum_max = equity.cummax()
        dd = equity / cum_max - 1.0
        max_dd = dd.min()
        years = (equity.index[-1] - equity.index[0]).days / 365.25
        cagr = equity.iloc[-1] ** (1/years) - 1 if years > 0 else np.nan
        ann_vol = pnl.std() * np.sqrt(len(pnl) / years) if years > 0 else np.nan
        sharpe = (pnl.mean() / (pnl.std() + 1e-9)) * np.sqrt(len(pnl) / years) if years > 0 else np.nan

        print("\n--- RISK METRICS ---")
        print(f"Strategy CAGR:        {cagr*100:8.2f}%")
        print(f"Max Drawdown:           {max_dd*100:8.2f}%")
        print(f"Annualized Volatility: {ann_vol*100:8.2f}%")
        print(f"Approximate Sharpe Ratio:           {sharpe:8.2f}")

        # Charts
        fig, axes = plt.subplots(3, 1, figsize=(14, 10))
        axes[0].plot(results.index, results['Equity'],
                     label='Strategy', color='blue', lw=2)
        axes[0].plot(results.index, results['BuyHold'],
                     label='Buy & Hold', color='gray', alpha=0.5, ls='--')
        axes[0].axhline(1.0, color='black', lw=0.5)
        axes[0].set_yscale('log')
        axes[0].set_title(f"Equity Curve (log scale) | WinRate: {win_rate:.1f}%")
        axes[0].legend()
        axes[0].grid(alpha=0.3, which='both')

        axes[1].plot(results.index, results['Target_Vol']*100,
                     color='green', label='Optimal Target Volatility')
        axes[1].set_title("Target Volatility")
        axes[1].set_ylabel("Vol %")
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        if len(trades) > 0:
            sc = axes[2].scatter(trades.index, trades['Size']*100,
                                 c=trades['PnL'], cmap='RdYlGn',
                                 alpha=0.6, s=20)
            axes[2].set_title("Position Sizing (color = PnL)")
            axes[2].set_ylabel("Size %")
            axes[2].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()
    else:
        print(" No trades were generated by the backtest.")

except Exception:
    raise
